In [1]:
from nba_api.stats.endpoints import synergyplaytypes
import pandas as pd
import time
import numpy as np

In [2]:
PLAY_TYPES = [
    "Transition",
    "Isolation",
    "PRBallHandler",
    "PRRollman",
    "Postup",
    "Spotup",
    "Handoff",
    "Cut",
    "OffScreen",
    "OffRebound",
    "Misc"
]

TYPE_GROUPINGS = ["Offensive", "Defensive"]

SEASONS = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26",
]

In [3]:
def fetch_synergy_stats(season):
    all_data = []

    for grouping in TYPE_GROUPINGS:
        for play_type in PLAY_TYPES:

            print(f"Fetching {grouping}: {play_type}")

            endpoint = synergyplaytypes.SynergyPlayTypes(
                league_id="00",
                season=season,
                season_type_all_star="Regular Season",
                per_mode_simple="Totals",
                player_or_team_abbreviation="P",
                type_grouping_nullable=grouping,
                play_type_nullable=play_type,
                timeout=60
            )

            df = endpoint.get_data_frames()[0]

            if not df.empty:
                df["SEASON"] = season
                all_data.append(df)

            time.sleep(0.6)

    if not all_data:
        return pd.DataFrame()

    return pd.concat(all_data, ignore_index=True)

In [5]:
for SEASON in SEASONS:
  synergy_raw = fetch_synergy_stats(SEASON)
  synergy_raw.to_csv(
    f"basketball_data/synergy_raw_{SEASON}.csv",
    index=False
  )

Fetching Offensive: Transition
Fetching Offensive: Isolation
Fetching Offensive: PRBallHandler
Fetching Offensive: PRRollman
Fetching Offensive: Postup
Fetching Offensive: Spotup
Fetching Offensive: Handoff
Fetching Offensive: Cut
Fetching Offensive: OffScreen
Fetching Offensive: OffRebound
Fetching Offensive: Misc
Fetching Defensive: Transition
Fetching Defensive: Isolation
Fetching Defensive: PRBallHandler
Fetching Defensive: PRRollman
Fetching Defensive: Postup
Fetching Defensive: Spotup
Fetching Defensive: Handoff
Fetching Defensive: Cut
Fetching Defensive: OffScreen
Fetching Defensive: OffRebound
Fetching Defensive: Misc
Fetching Offensive: Transition
Fetching Offensive: Isolation
Fetching Offensive: PRBallHandler
Fetching Offensive: PRRollman
Fetching Offensive: Postup
Fetching Offensive: Spotup
Fetching Offensive: Handoff
Fetching Offensive: Cut
Fetching Offensive: OffScreen
Fetching Offensive: OffRebound
Fetching Offensive: Misc
Fetching Defensive: Transition
Fetching Defensive

In [4]:
def create_synergy_features(df):

    df = df[
        ["PLAYER_ID", "SEASON", "TYPE_GROUPING",
         "PLAY_TYPE", "POSS"]
    ].copy()

    df["POSS"] = pd.to_numeric(df["POSS"], errors="coerce")

    # Normalize grouping names
    df["TYPE_GROUPING"] = (
        df["TYPE_GROUPING"].str.upper()
    )

    # Combine duplicate player-season-playtype records
    df = (
        df.groupby(
            ["PLAYER_ID", "SEASON",
             "TYPE_GROUPING", "PLAY_TYPE"],
            as_index=False
        )["POSS"]
        .sum()
    )

    # Calculate total possessions separately for offense and defense
    df["TOTAL_POSS"] = (
        df.groupby(
            ["PLAYER_ID", "SEASON", "TYPE_GROUPING"]
        )["POSS"].transform("sum")
    )

    # Calculate possession percentage
    df["POSS_PCT"] = (
        df["POSS"] / df["TOTAL_POSS"].replace(0, pd.NA)
    ).fillna(0)

    # Create feature names
    df["FEATURE"] = (
        "SYNERGY_"
        + df["TYPE_GROUPING"]
        + "_"
        + df["PLAY_TYPE"].str.upper()
        + "_PCT"
    )

    # Pivot so every play type becomes its own feature
    features = df.pivot(
        index=["PLAYER_ID", "SEASON"],
        columns="FEATURE",
        values="POSS_PCT"
    ).fillna(0)

    features.columns.name = None

    return features.reset_index()

In [5]:
df_feats = pd.DataFrame()
for SEASON in SEASONS:
  df = pd.read_csv(f"basketball_data/synergy_raw_{SEASON}.csv")
  df_feats = pd.concat([df_feats, create_synergy_features(df)], ignore_index=True)

df_feats.head()

,PLAYER_ID,SEASON,SYNERGY_DEFENSIVE_HANDOFF_PCT,SYNERGY_DEFENSIVE_ISOLATION_PCT,SYNERGY_DEFENSIVE_OFFSCREEN_PCT,SYNERGY_DEFENSIVE_POSTUP_PCT,SYNERGY_DEFENSIVE_PRBALLHANDLER_PCT,SYNERGY_DEFENSIVE_PRROLLMAN_PCT,SYNERGY_DEFENSIVE_SPOTUP_PCT,SYNERGY_OFFENSIVE_CUT_PCT,SYNERGY_OFFENSIVE_HANDOFF_PCT,SYNERGY_OFFENSIVE_ISOLATION_PCT,SYNERGY_OFFENSIVE_MISC_PCT,SYNERGY_OFFENSIVE_OFFREBOUND_PCT,SYNERGY_OFFENSIVE_OFFSCREEN_PCT,SYNERGY_OFFENSIVE_POSTUP_PCT,SYNERGY_OFFENSIVE_PRBALLHANDLER_PCT,SYNERGY_OFFENSIVE_PRROLLMAN_PCT,SYNERGY_OFFENSIVE_SPOTUP_PCT,SYNERGY_OFFENSIVE_TRANSITION_PCT
0,2544,2021-22,0.072072,0.121622,0.060811,0.058559,0.132883,0.078829,0.475225,0.060155,0.029107,0.154592,0.048512,0.029754,0.021992,0.107374,0.175938,0.047219,0.098318,0.227038
1,2546,2021-22,0.059621,0.170732,0.037940,0.116531,0.089431,0.056911,0.468835,0.033175,0.021327,0.071090,0.047393,0.046209,0.046209,0.172986,0.028436,0.056872,0.313981,0.162322
2,2730,2021-22,0.000000,0.194631,0.000000,0.167785,0.000000,0.140940,0.496644,0.311927,0.000000,0.000000,0.100917,0.259939,0.000000,0.058104,0.000000,0.180428,0.048930,0.039755
3,2738,2021-22,0.080247,0.092593,0.000000,0.074074,0.376543,0.000000,0.376543,0.147541,0.000000,0.000000,0.172131,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.180328
4,2772,2021-22,0.000000,0.123596,0.123596,0.000000,0.303371,0.000000,0.449438,0.116505,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.106796,0.592233,0.184466


In [6]:
df_feats.to_csv(
  f"basketball_data/play_type_pct.csv",
  index=False
)